In [ ]:
import numpy as np
# import pyvista as pv
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt



u_grid = np.load('data_u.npy')
v_grid = np.load('data_v.npy')
w_grid = np.load('data_w.npy')
t_grid = np.load('data_t.npy') 
print("Data Loaded.")


Data Loaded.


In [ ]:
def get_velocity_at(position):
 
    x, y, z = position
    nx, ny, nz = u_grid.shape
    if x < 0 or x >= nx-1 or y < 0 or y >= ny-1 or z < 0 or z >= nz-1:
        return np.array([0.0, 0.0, 0.0])
        
    x0, y0, z0 = int(x), int(y), int(z)
    x1, y1, z1 = x0 + 1, y0 + 1, z0 + 1
    xd, yd, zd = x-x0, y-y0, z-z0
    
    def interp(grid):
        c00 = grid[x0, y0, z0]*(1-xd) + grid[x1, y0, z0]*xd  
        c01 = grid[x0, y0, z1]*(1-xd) + grid[x1, y0, z1]*xd
        c10 = grid[x0, y1, z0]*(1-xd) + grid[x1, y1, z0]*xd
        c11 = grid[x0, y1, z1]*(1-xd) + grid[x1, y1, z1]*xd
        c0 = c00*(1-yd) + c10*yd 
        c1 = c01*(1-yd) + c11*yd
        return c0*(1-zd) + c1*zd

    return np.array([interp(u_grid), interp(v_grid), interp(w_grid) * 50])

In [3]:
def compute_streamline(seed, step_size=1, max_steps=500):
    path = [seed]
    curr = np.array(seed)
    
    for _ in range(max_steps):
        vel = get_velocity_at(curr)
        mag = np.linalg.norm(vel)
        if mag == 0: break 
        
        direction = vel / mag
        # print(direction) 
        step_vector = np.array([direction[1], direction[0], direction[2]])
        
        next_pos = curr + (step_vector * step_size)
        path.append(next_pos)
        curr = next_pos
        
    return np.array(path)

In [ ]:
def plot_interactive(lat_seed, lon_seed, alt_seed, view_elev, view_azim):
    
    
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    
    temp_slice = t_grid[:, :, 0]
    X, Y = np.meshgrid(np.arange(temp_slice.shape[1]), np.arange(temp_slice.shape[0]))
    ax.contourf(X, Y, temp_slice, zdir='z', offset=0, cmap='terrain', alpha=0.6)
    
    
    seed = [lat_seed, lon_seed, alt_seed]
    streamline = compute_streamline(seed, step_size=1.0)
    
    if len(streamline) > 1:
        ax.plot(streamline[:, 1], streamline[:, 0], streamline[:, 2], color='black', linewidth=1)
        ax.scatter(seed[1], seed[0], seed[2], color='red', s=50, label='Start')
        ax.scatter(streamline[-1, 1], streamline[-1, 0], streamline[-1, 2], color='blue', s=50)

    
    ax.set_title(f"Streamline")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_zlabel("Altitude")
    ax.set_xlim(0, 360); ax.set_ylim(0, 180); ax.set_zlim(0, 50)
    
   
    ax.view_init(elev=view_elev, azim=view_azim)
    
    
    plt.show()


print("Creating interactive tool.")

interact(plot_interactive, 
         
         lat_seed=IntSlider(min=10, max=170, step=1, value=90, description='Lat (N/S)'),
         lon_seed=IntSlider(min=10, max=350, step=1, value=180, description='Lon (E/W)'),
         alt_seed=IntSlider(min=1, max=45, step=1, value=15, description='Height'),
         
         
         view_elev=IntSlider(min=0, max=90, step=5, value=60, description='View Up/Down'),
         view_azim=IntSlider(min=-180, max=180, step=10, value=-90, description='Rotate View'))

Creating interactive tool.


interactive(children=(IntSlider(value=90, description='Lat (N/S)', max=170, min=10), IntSlider(value=180, desc…

<function __main__.plot_interactive(lat_seed, lon_seed, alt_seed, view_elev, view_azim)>